In [1]:
import sys
import os
import json
import subprocess
from pathlib import Path
import threading
import json
import py3Dmol
from train.train_config import (
    TrainConfig,
    TrainingParams,
    CheckpointParams,
    LoggingParams,
    LoaderConfig,
    NoiseScheduleParams,
)

from sample.sample_config import (
    SampleConfig,
    SamplerParams,
    GenerationParams,
    SampleCheckpointParams,
    SampleOutputParams,
)

PALLATOM_ROOT = Path.cwd().resolve()
if str(PALLATOM_ROOT) not in sys.path:
    sys.path.insert(0, str(PALLATOM_ROOT))

PALLATOM_ROOT

PosixPath('/workspaces/diffusion/pallatom')

# PallAtom: Training & Analysis

1. **Configure** — tune hyperparameters and serialise `TrainConfig` to JSON
2. **Train** — launch `train_loop.py` as a subprocess
3. **Sample** — load the saved checkpoint and run EDM backbone sampling
4. **Visualise** — render sampled structures with py3Dmol

## 1 · Training Configuration

### A · Compute EDM data parameters for train dataset

In [2]:
def run_subprocess(cmd, out: dict):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PALLATOM_ROOT)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(PALLATOM_ROOT),
        env=env,
    )
    out["pid"] = proc.pid
    print(f"Subprocess PID: {proc.pid}")

    stdout, _ = proc.communicate()
    output_lines = stdout.strip().splitlines() if stdout else []

    if proc.returncode != 0:
        print([f"ERROR (exit {proc.returncode}):"] + output_lines)
        out["result"] = None
        return

    out["result"] = output_lines

In [ ]:
# 1. Define Paths & Command
script = PALLATOM_ROOT / "helpers" / "compute_EDM_data_params.py"
data_jsonl = PALLATOM_ROOT / "data" / "chain_set.jsonl"
splits_json = PALLATOM_ROOT / "data" / "chain_set_splits.json"

cmd = [sys.executable, "-u", str(script), "--data", str(data_jsonl), "--splits", str(splits_json)]

# 2. Execute Subprocess
edm_out = {}
print("Computing EDM parameters...")

threading.Thread(
    target=run_subprocess,
    args=(cmd, edm_out),
    daemon=True
).start()

Computing EDM parameters...
Subprocess PID: 35713


In [26]:
edm_out

{'pid': 35713,
 'result': ["/workspaces/diffusion/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.",
  '  super().__init__(loader)',
  'sigma_data = 19.2368',
  'sigma_max  = 42.3689',
  'sigma_min  = 3.807123',
  'P_mean     = 2.5416',
  'P_std      = 1.2048']}

In [ ]:
tcfg = TrainConfig(
    training=TrainingParams(num_epochs=500, pretrained_weights="pallatom_best_best.pt"),
    train_loader=LoaderConfig(max_seq_length=128, batch_size=4)
    )
tcfg

TrainConfig(training=TrainingParams(num_epochs=50, lr=0.0003, weight_decay=0.0001, grad_clip=2.0, pretrained_weights='pallatom_best_best.pt'), model=ModelParams(f_ref_dim=35, n_bins=39, c_atom=16, c_pair=16, c_res=32, c_atompair=2, K_unit=3), noise=NoiseScheduleParams(sigma_data=19.2368, sigma_max=42.3689, sigma_min=3.807123, P_mean=2.5416, P_std=1.2048), distogram_res=ResidueDistogramParams(min_dist=3.25, max_dist=50.75, n_bins=38, tok_emb_dim=32), distogram_atom=AtomDistogramParams(min_dist=0.0, max_dist=10.0, n_bins=22, tok_emb_dim=32), loss=LossParams(lam=1.0, alpha_0=0.25, alpha_1=1.0, alpha_2=0.5, alpha_3=0.5, alpha_4=1.0, gamma=0.99, smooth_lddt_cutoff=15), checkpoint=CheckpointParams(checkpoint_path='pallatom_best_best.pt', save_every=1), logging=LoggingParams(log_interval=1, use_wandb=True, wandb_project='pallatom-training'), loader=LoaderConfig(max_seq_length=128, batch_size=2), train_loader=LoaderConfig(max_seq_length=128, batch_size=2), test_loader=LoaderConfig(max_seq_leng

In [5]:
config_json_path = PALLATOM_ROOT / "train" / "run_config.json"
with open(config_json_path, "w") as _f:
    _f.write(json.dumps(tcfg.model_dump(), indent=2))

In [6]:
log_path = PALLATOM_ROOT / "train" / "logs.jsonl"

train_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "train" / "train_loop.py"),
            "--data",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
            "--splits",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
            "--config",      config_json_path,
            "--log_file",    log_path,
            "--num_workers", "0",
        ]

training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(train_cmd, training_out),
    daemon=True
).start()

Subprocess PID: 84797


In [8]:
ckpt_path       = str(PALLATOM_ROOT / tcfg.checkpoint.checkpoint_path)
ckpt_path

'/workspaces/diffusion/pallatom/pallatom_best_best.pt'

In [9]:
sample_output_path     = str(Path(config_json_path).with_name("samples.json"))
sample_cfg_path = str(Path(config_json_path).with_name("sample_config.json"))

scfg = SampleConfig(
    model=tcfg.model,
    noise=tcfg.noise,
    generation=GenerationParams(
        n_res=tcfg.test_loader.max_seq_length,
        n_samples=tcfg.test_loader.batch_size
    ),
    checkpoint=SampleCheckpointParams(checkpoint_path=ckpt_path),
    output=SampleOutputParams(output_path=sample_output_path),
)

sample_config_json_path = PALLATOM_ROOT / "train" / "sample_config.json"
with open(sample_config_json_path, "w") as _f:
    _f.write(json.dumps(scfg.model_dump(), indent=2))

In [ ]:
sample_log_path = PALLATOM_ROOT / "train" / "sample_logs.jsonl"
sample_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "sample" / "sampling.py"),
            "--config", sample_cfg_path,
            "--log_file", sample_log_path,
        ]
# does this not pipe logs out into a structlog
sample_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(sample_cmd, sample_out),
    daemon=True
).start()

Subprocess PID: 81167


In [11]:
sample_out

{'pid': 81167,
 'result': ['device=cpu',
  'Loading checkpoint: /workspaces/diffusion/pallatom/pallatom_best_best.pt',
  'Model loaded.',
  'Sampling 2 structure(s) in one batched call ...',
  'Structure 1/2 done.',
  'Structure 2/2 done.',
  'Wrote 2 PDB string(s) to /workspaces/diffusion/pallatom/train/samples.json']}

In [14]:
# lets load the pdb files from samples.json

# Open the file in read mode ('r')
with open('train/samples.json', 'r') as file:
    # Use json.load() to read and parse the file
    data = json.load(file)

# Now 'data' is a standard Python object (dict or list)
print(len(data))
pdb_str = data[1]
view = py3Dmol.view(
    width=600, height=600, linked=True , viewergrid=(1, 1))
view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
style = {"cartoon": {'color': 'spectrum'}}

view.addModelsAsFrames(pdb_str, viewer=(0, 0))
view.setStyle({'model': -1}, style, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

view.render()


# and then show them in py3dmol

2


3Dmol.js failed to load for some reason. Please check your browser console for error messages.